In [0]:
# Imports

from pyspark.sql.functions import col, trim, to_date, expr, when, current_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col

In [0]:
# Ler a Bronze V2
df_bronze = spark.table("workspace.drs_bronze.faixa_hetaria_padre_bernardo_22")

In [0]:
# Padronizando os nomes das colunas
df_faixa_hetaria = df_bronze \
    .withColumnRenamed("id_municipio", "municipality_id") \
    .withColumnRenamed("forma_declaracao_idade", "age_declaration_method") \
    .withColumnRenamed("sexo", "gender") \
    .withColumnRenamed("idade", "age") \
    .withColumnRenamed("idade_anos", "age_years") \
    .withColumnRenamed("grupo_idade", "age_group") \
    .withColumnRenamed("populacao", "population")
 
print(f"Total registros: {df_faixa_hetaria.count()}")


In [0]:
# Transformar/Convert colunas para o padarão para depois salvar a Silver V2

from pyspark.sql.functions import col, trim, to_date, expr, when, lower

df_silver = (
    df_faixa_hetaria
    .withColumn("age_years", col("age_years").cast("int"))
    .withColumn("age_declaration_method", lower(col("age_declaration_method")))
    .withColumn("population", col("population").cast("int"))
)

In [0]:
# criando o campo source_file
from pyspark.sql.functions import lit

df = df_silver.withColumn("source_file", lit("faixa_hetaria_padre_bernardo_22.csv"))

In [0]:
# Separando registros inválidos para quarentena

df_invalid = df.filter("""
    population IS NULL
    OR age_years IS NULL
    OR age_group IS NULL
""")
 
print(f"Total registros inválidos: {df_invalid.count()}")

In [0]:
# Salvando quarentena da Silver V2

df_invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.drs_silver.faixa_hetaria_padre_bernardo_22_quarantine")


In [0]:
# criando o campo ingestion_timestamp
from pyspark.sql.functions import current_timestamp

df_silver = df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# Filtrando valores válidos
from pyspark.sql.functions import col

df_silver_filtered = (
    df_silver
    .filter(
        (col("population").isNotNull()) &
        (col("age_years").isNotNull()) &
        (col("age_group").isNotNull())
    )
)
 
print(f"Total registros válidos: {df_silver_filtered.count()}")


In [0]:
# Criando o campo silver_processed_timestamp em df_silver_v2
from pyspark.sql.functions import current_timestamp

df_faixa_hetaria = df_silver_filtered \
    .withColumn("silver_processed_timestamp", current_timestamp()) \
    .withColumn("created_at", current_timestamp()) \
    .withColumn("updated_at", current_timestamp())

In [0]:
# Grantindo a deduplicação por chave de negócio

window_spec = Window.partitionBy(
    "municipality_id",
    "gender",
    "age_years",
    "age_group"
).orderBy(
    col("ingestion_timestamp").desc()
)
 
df_faixa_hetaria = (
    df_faixa_hetaria
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)
 
print(f"Total após deduplicação: {df_faixa_hetaria.count()}")

In [0]:
# Salvando a tabela Silver em uma staging table

df_faixa_hetaria.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.drs_silver.faixa_hetaria_padre_bernardo_22_staging")

In [0]:

# Craindo tabela final se não existir

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.drs_silver.faixa_hetaria_padre_bernardo_22
AS
SELECT * FROM workspace.drs_silver.faixa_hetaria_padre_bernardo_22_staging WHERE 1 = 0
""")

In [0]:
# Salvando tabela drs_silver_v2 com MERGE

spark.sql("""
MERGE INTO workspace.drs_silver.faixa_hetaria_padre_bernardo_22 AS target
USING workspace.drs_silver.faixa_hetaria_padre_bernardo_22_staging AS source
 
ON  target.municipality_id = source.municipality_id
AND target.gender           = source.gender
AND target.age_years        = source.age_years
AND target.age_group        = source.age_group
 
WHEN MATCHED THEN
UPDATE SET
    target.age_declaration_method    = source.age_declaration_method,
    target.age                       = source.age,
    target.population                = source.population,
    target.ingestion_timestamp       = source.ingestion_timestamp,
    target.source_file               = source.source_file,
    target.silver_processed_timestamp = source.silver_processed_timestamp,
    target.updated_at                = current_timestamp()
 
WHEN NOT MATCHED THEN
INSERT (
    municipality_id,
    age_declaration_method,
    gender,
    age,
    age_years,
    age_group,
    population,
    ingestion_timestamp,
    source_file,
    silver_processed_timestamp,
    created_at,
    updated_at
)
VALUES (
    source.municipality_id,
    source.age_declaration_method,
    source.gender,
    source.age,
    source.age_years,
    source.age_group,
    source.population,
    source.ingestion_timestamp,
    source.source_file,
    source.silver_processed_timestamp,
    current_timestamp(),
    current_timestamp()
)
""")

In [0]:
# Validando a tabela drs_silver_votacao_secao_2024_go

spark.sql("""
SELECT *
FROM workspace.drs_silver.faixa_hetaria_padre_bernardo_22
LIMIT 10
""")
 

In [0]:
# Verificar duplicidade pela chave do merge

spark.sql("""
SELECT
    municipality_id,
    gender,
    age_years,
    age_group,
    COUNT(*) AS qtd
FROM workspace.drs_silver.faixa_hetaria_padre_bernardo_22
GROUP BY municipality_id, gender, age_years, age_group
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""")

In [0]:
# validação quarentena
spark.sql("""
SELECT COUNT(*) AS total_invalid
FROM workspace.drs_silver.faixa_hetaria_padre_bernardo_22_quarantine
""")

In [0]:
# Data Quality checks da Silver V2
dq = spark.sql("""
SELECT
    SUM(CASE WHEN age_years   IS NULL THEN 1 ELSE 0 END) AS age_years_nulls,
    SUM(CASE WHEN population  IS NULL THEN 1 ELSE 0 END) AS population_nulls
FROM workspace.drs_silver.faixa_hetaria_padre_bernardo_22
""")

In [0]:
# Falhar notebook se houver erro crítico de qualidade
dq_row = dq.collect()[0]
 
if (
    dq_row["age_years_nulls"] > 0 or
    dq_row["population_nulls"] > 0
):
    raise Exception("Data Quality check failed: workspace.drs_silver.faixa_hetaria_padre_bernardo_22")
 
print("✅ Pipeline Silver concluído com sucesso!")
